# 02 · Paralelismo C++ / OpenMP y Docker Compose

**Objetivo:** ejecutar el mismo ETL en paralelo dentro de un contenedor y repartir particiones entre dos contenedores. El código de procesamiento y el reductor son C++.


In [1]:
from pathlib import Path
import sys, json
import pandas as pd
from PIL import Image

ROOT = Path.cwd()
if not (ROOT / "cpp/etl.cpp").exists():
    raise RuntimeError("Abrir y ejecutar el notebook desde la carpeta del proyecto")
sys.path.insert(0, str(ROOT / "scripts"))
OUT = ROOT / "resultados_experimentos"
entorno = json.loads((OUT / "entorno.json").read_text())
mediciones = pd.read_csv(OUT / "mediciones.csv")
resumen = pd.read_csv(OUT / "resumen_rendimiento.csv")
pd.set_option("display.max_columns", 20)
pd.set_option("display.float_format", lambda x: f"{x:.6f}")
print("Dataset SHA-256:", entorno["dataset_sha256"])
print("Compilación:", entorno["compilacion"])
print("Kubernetes:", entorno["kubernetes"])


Dataset SHA-256: ca65e96d26e1e7cb58b0b5ec1c642fbcb24926684077cbee5cf2864251ce9728
Compilación: g++ 13.3.0; -std=c++17 -O3 -Wall -Wextra -Wpedantic -fopenmp
Kubernetes: v1.34.13+k3s1


## 1. Memoria compartida con OpenMP

La preparación global resuelve duplicados. Después, `omp for schedule(static)` reparte las filas entre hilos. Cada hilo modifica filas distintas y la suma de edades inválidas usa `reduction`, por lo que no hay un contador compartido sin protección. El programa registra el tamaño real del equipo OpenMP.


In [2]:
fuente = (ROOT / "cpp/etl.cpp").read_text().splitlines()
inicio = next(i for i, linea in enumerate(fuente) if "#pragma omp parallel" in linea)
print("\n".join(fuente[inicio-3:inicio+12]))
display(resumen[resumen.configuracion.str.startswith("cpp_")][["configuracion", "hilos", "repeticiones", "etl_segundos_media"]])

    if (o.mode=="omp") {
        omp_set_dynamic(0);
        // Cada hilo modifica filas distintas; reducción para el contador común.
        #pragma omp parallel num_threads(o.threads) reduction(+:invalid) default(none) shared(t,stats,dates,age,count)
        {
            #pragma omp single
            stats.actual=omp_get_num_threads();
            #pragma omp for schedule(static)
            for (long long i=0;i<count;++i) invalid+=transform_row(t.rows[i],dates,age);
        }
    } else {
        for (auto& row:t.rows) invalid+=transform_row(row,dates,age);
    }
    stats.invalid_age=invalid;
    stats.output=t.rows.size();


,configuracion,hilos,repeticiones,etl_segundos_media
0,cpp_secuencial,1,5,0.201135
1,cpp_openmp_1,1,5,0.198650
2,cpp_openmp_2,2,5,0.170713
3,cpp_openmp_4,4,5,0.158650
4,cpp_openmp_8,8,5,0.164510


## 2. Imagen y ejecución Docker

La etapa de compilación reutiliza `jupyterlab_docker_master-openmp:latest`. La imagen final contiene el ejecutable, sus bibliotecas dinámicas y el dataset, y se ejecuta como usuario 1000. No instala Python para realizar el ETL. El binario de esa misma imagen se copia a Workbench para evitar comparar compilaciones distintas.

Ejecutar desde el anfitrión con Docker disponible:

```bash
docker build --network=none -f Dockerfile.cpp -t etl-distribuido-cpp:local .
mkdir -p resultados_docker_cpp
docker run --rm --network=none --cpus=4 --memory=512m   --mount type=bind,src="$PWD/resultados_docker_cpp",dst=/resultados   etl-distribuido-cpp:local
```


In [3]:
print((ROOT / "Dockerfile.cpp").read_text())
print((OUT / "ejecuciones/docker_openmp_4_05/ejecucion.log").read_text())
display(pd.DataFrame([json.loads((OUT / "ejecuciones/docker_openmp_4_05/validacion.json").read_text())]))

# Reutiliza el entorno C/C++ y OpenMP que ya existe en este equipo.
ARG ETL_CPP_BASE=jupyterlab_docker_master-openmp:latest
FROM ${ETL_CPP_BASE} AS build
WORKDIR /opt/etl
COPY cpp/etl.cpp cpp/etl.cpp
RUN g++ -std=c++17 -O3 -Wall -Wextra -Wpedantic -fopenmp cpp/etl.cpp -o /usr/local/bin/etl
FROM scratch
COPY --from=build /usr/local/bin/etl /usr/local/bin/etl
COPY --from=build /lib/x86_64-linux-gnu/libstdc++.so.6 /lib/x86_64-linux-gnu/libstdc++.so.6
COPY --from=build /lib/x86_64-linux-gnu/libgcc_s.so.1 /lib/x86_64-linux-gnu/libgcc_s.so.1
COPY --from=build /lib/x86_64-linux-gnu/libgomp.so.1 /lib/x86_64-linux-gnu/libgomp.so.1
COPY --from=build /lib/x86_64-linux-gnu/libm.so.6 /lib/x86_64-linux-gnu/libm.so.6
COPY --from=build /lib/x86_64-linux-gnu/libc.so.6 /lib/x86_64-linux-gnu/libc.so.6
COPY --from=build /lib64/ld-linux-x86-64.so.2 /lib64/ld-linux-x86-64.so.2
COPY datos/COVID19MEXICO.csv /datos/COVID19MEXICO.csv
USER 1000:1000
ENTRYPOINT ["/usr/local/bin/etl"]
CMD ["--input", "/datos/COVID

,filas,columnas,coincide_referencia_python,sha256_csv,auditoria_coincide_referencia
0,108733,42,True,7ac454c8bc4420dd7f4d36fd168820f1b2e5f608629d95...,True


## 3. Dos trabajadores y unión con Compose

El hash estable FNV-1a del `ID_REGISTRO`, antes de recortar el texto, asigna cada registro a una de dos particiones. Los identificadores repetidos quedan en el mismo trabajador. Cada trabajador usa dos hilos y exporta índices originales; el reductor C++ espera las dos terminaciones correctas y reúne las filas en el orden de entrada.

Cada trabajador lee todo el CSV para inferir el mismo esquema; solamente transforma su partición. Esta lectura repetida y la unión añaden costo, que se incluye en la evaluación.

```bash
mkdir -p resultados_docker_cpp
docker compose -f compose.cpp.yaml up --no-build --force-recreate
docker compose -f compose.cpp.yaml down
```


In [4]:
print((ROOT / "compose.cpp.yaml").read_text())
carpeta = OUT / "ejecuciones/compose_2x2_05"
auditorias = [json.loads((carpeta / f"parte_{i}.json").read_text()) for i in range(2)]
display(pd.DataFrame(auditorias)[["particion", "filas_entrada", "filas_salida", "hilos_utilizados", "etl_segundos"]])
assert sum(a["filas_salida"] for a in auditorias) == 108733
assert sum(a["hilos_utilizados"] for a in auditorias) == 4
display(pd.DataFrame([json.loads((carpeta / "validacion.json").read_text())]))
print((carpeta / "ejecucion.log").read_text())

name: etl-cpp
x-worker: &worker
  image: etl-distribuido-cpp:local
  network_mode: none
  user: "${ETL_UID:-1000}:${ETL_GID:-1000}"
  cpus: 2
  mem_limit: 512m
  volumes:
    - ${ETL_RESULTS_DIR:-./resultados_docker_cpp}:/resultados

services:
  parte0:
    <<: *worker
    command: ["--input", "/datos/COVID19MEXICO.csv", "--output", "/resultados/parte_0.csv", "--audit", "/resultados/parte_0.json", "--mode", "omp", "--threads", "2", "--shard-index", "0", "--shard-count", "2"]
  parte1:
    <<: *worker
    command: ["--input", "/datos/COVID19MEXICO.csv", "--output", "/resultados/parte_1.csv", "--audit", "/resultados/parte_1.json", "--mode", "omp", "--threads", "2", "--shard-index", "1", "--shard-count", "2"]
  reunir:
    <<: *worker
    cpus: 1
    depends_on:
      parte0:
        condition: service_completed_successfully
      parte1:
        condition: service_completed_successfully
    command: ["--merge-dir", "/resultados", "--shard-count", "2", "--output", "/resultados/COVID19MEXI

,particion,filas_entrada,filas_salida,hilos_utilizados,etl_segundos
0,0,54118,54118,2,0.140587
1,1,54615,54615,2,0.143466


,filas,columnas,coincide_referencia_python,sha256_csv,auditoria_coincide_referencia
0,108733,42,True,7ac454c8bc4420dd7f4d36fd168820f1b2e5f608629d95...,True


 Container etl-cpp-experimento-parte0-1 Creating 
 Container etl-cpp-experimento-parte1-1 Creating 
 Container etl-cpp-experimento-parte1-1 Created 
 Container etl-cpp-experimento-parte0-1 Created 
 Container etl-cpp-experimento-reunir-1 Creating 
 Container etl-cpp-experimento-reunir-1 Created 
Attaching to parte0-1, parte1-1, reunir-1
 Container etl-cpp-experimento-parte1-1 Starting 
 Container etl-cpp-experimento-parte0-1 Starting 
 Container etl-cpp-experimento-parte1-1 Started 
 Container etl-cpp-experimento-parte0-1 Started 
 Container etl-cpp-experimento-parte0-1 Waiting 
 Container etl-cpp-experimento-parte1-1 Waiting 
parte0-1  | C++ omp | hilos=2 | partición=0/2 | filas=54118 | ETL=0.140587 s | total=0.970902 s
parte1-1  | C++ omp | hilos=2 | partición=1/2 | filas=54615 | ETL=0.143466 s | total=1.00996 s
 Container etl-cpp-experimento-parte0-1 Exited 

parte0-1 exited with code 0

parte1-1 exited with code 0
 Container etl-cpp-experimento-parte1-1 Exited 
 Container etl-cpp-e

## Resultado del avance 2

Se comprobaron dos niveles de paralelismo: hilos OpenMP dentro de un proceso y particiones distintas entre contenedores. La salida de ambos coincide con la referencia secuencial y conserva el orden. La evaluación repetida de sus tiempos está en el notebook 04.


## Fuentes técnicas

- [OpenMP: distribución de iteraciones](https://www.openmp.org/spec-html/5.2/openmpse66.html).
- [GCC: compilación con `-fopenmp`](https://gcc.gnu.org/projects/gomp/).
- [Kubernetes: Jobs paralelos e indexados](https://kubernetes.io/docs/concepts/workloads/controllers/job/).
- [Cornell: Ley de Amdahl](https://cvw.cac.cornell.edu/parallel/efficiency/amdahls-law).

Los resultados numéricos proceden exclusivamente de los CSV, auditorías y registros de esta entrega.
